In [1]:
def parse_result_name(path, aggregators=None):
    from pathlib import Path

    path = Path(path)
    aggregators = aggregators or ["fedavg", "median", "trimmedmean", "krum", "multikrum"]
    stem = path.stem
    source_dir = path.parent.name

    if source_dir == "no_attacks":
        aggregator, nnm_mode, seed = stem.rsplit("_", 2)
        return aggregator, "no_attack", nnm_mode, int(seed)

    if source_dir in {"classic_attacks", "pca_attacks"}:
        prefix, nnm_mode, seed = stem.rsplit("_", 2)
        for aggregator in aggregators:
            marker = f"{aggregator}_"
            if prefix.startswith(marker):
                attack = prefix[len(marker):]
                return aggregator, attack, nnm_mode, int(seed)

    raise ValueError(f"Cannot parse result filename: {path}")


In [2]:
def load_all_results(
    result_dirs=("../results/cifar10/no_attacks", "../results/cifar10/classic_attacks", "../results/cifar10/pca_attacks"),
    aggregators=("fedavg", "median", "trimmedmean", "krum", "multikrum"),
    seeds=(40, 60, 11),
    max_round=50,
    random_guess_acc=0.1,
    exclude_nnm_sign_flip=True,
):
    from pathlib import Path
    import numpy as np
    import pandas as pd

    metric_columns = [
        "test_accuracy",
        "malicious_to_benign_mean_distance",
        "malicious_to_benign_mean_cosine",
        "attack_aggregation_shift",
    ]
    full_rounds = pd.DataFrame({"round": range(max_round + 1)})
    all_runs = []

    for result_dir in [Path(p) for p in result_dirs]:
        if not result_dir.exists():
            print(f"Missing directory: {result_dir}")
            continue

        for path in sorted(result_dir.glob("*.csv")):
            aggregator, attack, nnm_mode, seed = parse_result_name(path, list(aggregators))
            if aggregator not in aggregators or seed not in seeds:
                continue
            if exclude_nnm_sign_flip and nnm_mode == "nnm" and attack in {"sign_flip", "signflip"}:
                continue

            df = pd.read_csv(path)
            df["round"] = pd.to_numeric(df["round"], errors="coerce")
            df = df.dropna(subset=["round"])
            df["round"] = df["round"].astype(int)

            keep_cols = ["round"]
            for col in metric_columns:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                else:
                    df[col] = np.nan
                keep_cols.append(col)

            df = full_rounds.merge(df[keep_cols], on="round", how="left")
            df["test_accuracy_filled"] = df["test_accuracy"].fillna(random_guess_acc)
            df["aggregator"] = aggregator
            df["attack"] = attack
            df["nnm_mode"] = nnm_mode
            df["seed"] = seed
            df["file"] = path.name
            df["source_dir"] = result_dir.name
            all_runs.append(df)

    if not all_runs:
        raise ValueError("No result CSV files were loaded.")

    return pd.concat(all_runs, ignore_index=True)


In [3]:
def add_plot_labels(all_results):
    defense_labels = {
        "fedavg": "FedAvg",
        "median": "Median",
        "trimmedmean": "TrimmedMean",
        "krum": "Krum",
        "multikrum": "MultiKrum",
    }
    attack_labels = {
        "no_attack": "No Attack",
        "alie_optim": "ALIE",
        "min_max": "MinMax",
        "min_sum": "MinSum",
        "sign_flip": "SF",
        "stdk2_p0": "PCA k=2",
        "stdk3_p0": "PCA k=3",
    }

    labeled = all_results.copy()
    labeled["defense"] = labeled["aggregator"].map(defense_labels).fillna(labeled["aggregator"])
    labeled["attack_label"] = labeled["attack"].map(attack_labels).fillna(labeled["attack"])
    return labeled


In [4]:
def summarize_last_n(all_results, metrics=None, last_n=5):
    metrics = metrics or [
        "test_accuracy_filled",
        "malicious_to_benign_mean_distance",
        "malicious_to_benign_mean_cosine",
        "attack_aggregation_shift",
    ]

    run_summary = (
        all_results
        .sort_values(["aggregator", "attack", "nnm_mode", "seed", "round"])
        .groupby(["aggregator", "attack", "nnm_mode", "seed"], as_index=False)
        .tail(last_n)
        .groupby(["aggregator", "attack", "nnm_mode", "seed"], as_index=False)[metrics]
        .mean()
    )

    avg_summary = (
        run_summary
        .groupby(["aggregator", "attack", "nnm_mode"], as_index=False)[metrics]
        .agg(["mean", "std", "count"])
    )
    avg_summary.columns = ["_".join(col).strip("_") for col in avg_summary.columns.to_flat_index()]
    avg_summary = avg_summary.reset_index()

    avg_summary["test_accuracy_filled_mean"] *= 100
    avg_summary["test_accuracy_filled_std"] = avg_summary["test_accuracy_filled_std"].fillna(0) * 100
    for metric in metrics:
        std_col = f"{metric}_std"
        if std_col in avg_summary.columns:
            avg_summary[std_col] = avg_summary[std_col].fillna(0)

    return run_summary, add_plot_labels(avg_summary)


In [5]:
def make_metric_table(avg_summary, metric, nnm_mode):
    import pandas as pd

    defense_order = ["FedAvg", "Median", "TrimmedMean", "Krum", "MultiKrum"]
    attack_order = ["No Attack", "ALIE", "MinMax", "MinSum", "SF", "PCA k=2", "PCA k=3"]
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    mode_df = avg_summary[avg_summary["nnm_mode"] == nnm_mode].copy()
    mode_df["cell"] = mode_df.apply(
        lambda r: "" if pd.isna(r.get(mean_col)) else f"{r[mean_col]:.4f} ± {r[std_col]:.4f}",
        axis=1,
    )

    present_defenses = [d for d in defense_order if d in set(mode_df["defense"])]
    present_attacks = [a for a in attack_order if a in set(mode_df["attack_label"])]

    return (
        mode_df
        .pivot(index="defense", columns="attack_label", values="cell")
        .reindex(index=present_defenses, columns=present_attacks)
        .rename_axis(index="Defense", columns=None)
    )


In [6]:
def build_mean_curve(all_results, metric):
    curve_df = add_plot_labels(all_results).dropna(subset=["round"]).copy()
    curve_df["round"] = curve_df["round"].astype(int)

    mean_curve = (
        curve_df
        .groupby(["nnm_mode", "defense", "attack_label", "round"], as_index=False)[metric]
        .mean()
    )

    if metric == "test_accuracy_filled":
        mean_curve[metric] = mean_curve[metric] * 100

    return mean_curve


In [7]:
def robust_ylim(values, metric, clip_quantile=0.95):
    import numpy as np

    values = values.replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return None

    if metric == "test_accuracy_filled":
        return 0, 100
    if metric == "malicious_to_benign_mean_cosine":
        return -1.05, 1.05

    y_min = values.quantile(1 - clip_quantile)
    y_max = values.quantile(clip_quantile)
    if y_min == y_max:
        y_min = values.min()
        y_max = values.max()
    if y_min == y_max:
        y_min -= 1
        y_max += 1

    pad = 0.05 * (y_max - y_min)
    return y_min - pad, y_max + pad


In [8]:
def plot_metric_curves(
    all_results,
    metric,
    metric_label,
    plot_dir="../results/cifar10/plots/diagnostics",
    clip_quantile=0.95,
    show=True,
):
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np

    defense_order = ["FedAvg", "Median", "TrimmedMean", "Krum", "MultiKrum"]
    attack_order = ["No Attack", "ALIE", "MinMax", "MinSum", "SF", "PCA k=2", "PCA k=3"]
    nnm_order = ["raw", "nnm"]
    highlight_attacks = {"PCA k=2", "PCA k=3"}
    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    mean_curve = build_mean_curve(all_results, metric)
    mean_curve[metric] = mean_curve[metric].replace([np.inf, -np.inf], np.nan)
    ylim = robust_ylim(mean_curve[metric], metric, clip_quantile)

    present_defenses = [d for d in defense_order if d in set(mean_curve["defense"])]
    present_attacks = [a for a in attack_order if a in set(mean_curve["attack_label"])]
    present_nnm_modes = [m for m in nnm_order if m in set(mean_curve["nnm_mode"])]
    saved_paths = []

    for nnm_mode in present_nnm_modes:
        for defense in present_defenses:
            subset = mean_curve[(mean_curve["nnm_mode"] == nnm_mode) & (mean_curve["defense"] == defense)]
            if subset.dropna(subset=[metric]).empty:
                continue

            fig, ax = plt.subplots(figsize=(7, 4.5))
            for attack in present_attacks:
                attack_df = subset[subset["attack_label"] == attack].sort_values("round").dropna(subset=[metric])
                if attack_df.empty:
                    continue

                is_highlight = attack in highlight_attacks
                ax.plot(
                    attack_df["round"],
                    attack_df[metric],
                    label=attack,
                    linewidth=3.0 if is_highlight else 1.3,
                    alpha=1.0 if is_highlight else 0.45,
                    zorder=3 if is_highlight else 1,
                )

            ax.set_title(f"{defense} | {nnm_mode.upper()} | {metric_label}")
            ax.set_xlabel("Round")
            ax.set_ylabel(metric_label)
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False, ncol=2)
            fig.tight_layout()

            file_defense = defense.lower().replace(" ", "_")
            save_path = plot_dir / f"{file_defense}_{nnm_mode}_{metric}.png"
            fig.savefig(save_path, dpi=300, bbox_inches="tight")
            saved_paths.append(save_path)
            if show:
                plt.show()
            else:
                plt.close(fig)

    return saved_paths


In [9]:
def run_analysis(last_n=5, clip_quantile=0.95, show=True):
    metrics = [
        "test_accuracy_filled",
        "malicious_to_benign_mean_distance",
        "malicious_to_benign_mean_cosine",
        "attack_aggregation_shift",
    ]
    metric_labels = {
        "test_accuracy_filled": "Mean Accuracy (%)",
        "malicious_to_benign_mean_distance": "Malicious to Benign Mean Distance",
        "malicious_to_benign_mean_cosine": "Malicious to Benign Mean Cosine",
        "attack_aggregation_shift": "Attack Aggregation Shift",
    }

    all_results = load_all_results()
    run_summary, avg_summary = summarize_last_n(all_results, metrics=metrics, last_n=last_n)
    tables = {
        (metric, nnm_mode): make_metric_table(avg_summary, metric, nnm_mode)
        for metric in metrics
        for nnm_mode in ["raw", "nnm"]
    }
    saved_paths = {
        metric: plot_metric_curves(
            all_results,
            metric,
            metric_labels[metric],
            clip_quantile=clip_quantile,
            show=show,
        )
        for metric in metrics
    }

    return {
        "all_results": all_results,
        "run_summary": run_summary,
        "avg_summary": avg_summary,
        "tables": tables,
        "saved_paths": saved_paths,
    }
